# Projection and the hat matrix

Least squares is orthogonal projection of $y$ onto the column space of $X$. The hat matrix is the matrix of that projection. Its diagonal entries, the leverages, say how much each observation can pull the fit.


## The projector

If $\widehat\beta = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$, the fitted vector is

$$
\widehat y = X\widehat\beta = Hy, \qquad H = X(X^{\mathsf T}X)^{-1}X^{\mathsf T}.
$$

Two algebraic facts follow at once. $H$ is symmetric, and

$$
H^2 = X(X^{\mathsf T}X)^{-1}X^{\mathsf T}X(X^{\mathsf T}X)^{-1}X^{\mathsf T} = H.
$$

So $H$ is the orthogonal projector onto the column space of $X$. The residual vector $e = (I - H)y$ satisfies $X^{\mathsf T}e = 0$: residuals are orthogonal to every column of $X$. Also $H$ has eigenvalues $0$ or $1$, and

$$
\operatorname{tr}(H) = \operatorname{tr}\bigl((X^{\mathsf T}X)^{-1}X^{\mathsf T}X\bigr) = p,
$$

the number of columns of $X$. When a column of ones is present, $H\mathbf 1 = \mathbf 1$, so every row of $H$ sums to $1$.


## The three-point projector, by hand

From the design-matrix lesson,

$$
(X^{\mathsf T}X)^{-1} = \begin{bmatrix} 7/3 & -1 \\ -1 & 1/2 \end{bmatrix}.
$$

The leverage of the first observation is the first diagonal entry. The first column of $X^{\mathsf T}$ is $(1, 1)^{\mathsf T}$, and

$$
(X^{\mathsf T}X)^{-1}\begin{bmatrix} 1 \\ 1 \end{bmatrix}
= \begin{bmatrix} 7/3 - 1 \\ -1 + 1/2 \end{bmatrix}
= \begin{bmatrix} 4/3 \\ -1/2 \end{bmatrix}.
$$

The first row of $X$ is $[1,\ 1]$, so

$$
h_{11} = \begin{bmatrix} 1 & 1 \end{bmatrix}\begin{bmatrix} 4/3 \\ -1/2 \end{bmatrix}
= \dfrac{4}{3} - \dfrac{1}{2} = \dfrac{5}{6}.
$$

The same multiplication for the other entries produces

$$
H = \begin{bmatrix}
5/6 & 1/3 & -1/6 \\
1/3 & 1/3 & 1/3 \\
-1/6 & 1/3 & 5/6
\end{bmatrix}.
$$

Checks that are visible without another page of products:

$$
\operatorname{tr}(H) = \dfrac{5}{6} + \dfrac{1}{3} + \dfrac{5}{6} = 2 = p,
$$

and each row sums to $1$. For simple regression there is also a direct formula,

$$
h_{ii} = \dfrac{1}{n} + \dfrac{(x_i - \bar x)^2}{S_{xx}}.
$$

With $n = 3$, $\bar x = 2$, and $S_{xx} = 2$,

$$
h_{11} = \dfrac{1}{3} + \dfrac{1}{2} = \dfrac{5}{6}, \qquad
h_{22} = \dfrac{1}{3}, \qquad
h_{33} = \dfrac{5}{6}.
$$

The endpoints, farther from $\bar x$, have the larger leverages.


In [1]:
# Build H from its formula and compare it with the hand matrix.
from fractions import Fraction
import numpy as np
X = np.array([
    [Fraction(1), Fraction(1)],
    [Fraction(1), Fraction(2)],
    [Fraction(1), Fraction(3)],
])
inverse = np.array([
    [Fraction(7, 3), Fraction(-1)],
    [Fraction(-1), Fraction(1, 2)],
])
H = X @ inverse @ X.T
hand = np.array([
    [Fraction(5, 6), Fraction(1, 3), Fraction(-1, 6)],
    [Fraction(1, 3), Fraction(1, 3), Fraction(1, 3)],
    [Fraction(-1, 6), Fraction(1, 3), Fraction(5, 6)],
])
y = np.array([Fraction(2), Fraction(3), Fraction(5)])
print("H")
print(H)
print("trace", np.trace(H))
print("row sums", H.sum(axis=1))
print("fitted", H @ y)
print("X' residual", X.T @ (y - H @ y))
assert np.all(H == hand)
assert np.all(H @ H == H)
assert np.trace(H) == 2
assert np.all(H.sum(axis=1) == 1)
assert np.all(X.T @ (y - H @ y) == 0)


H
[[Fraction(5, 6) Fraction(1, 3) Fraction(-1, 6)]
 [Fraction(1, 3) Fraction(1, 3) Fraction(1, 3)]
 [Fraction(-1, 6) Fraction(1, 3) Fraction(5, 6)]]
trace 2
row sums [Fraction(1, 1) Fraction(1, 1) Fraction(1, 1)]
fitted [Fraction(11, 6) Fraction(10, 3) Fraction(29, 6)]
X' residual [Fraction(0, 1) Fraction(0, 1)]


## Leverages on the five-point line

There $n = 5$, $\bar x = 3$, and $S_{xx} = 10$. The formula gives

$$
\begin{aligned}
h(1) &= \dfrac{1}{5} + \dfrac{4}{10} = \dfrac{3}{5}, \\
h(2) &= \dfrac{1}{5} + \dfrac{1}{10} = \dfrac{3}{10}, \\
h(3) &= \dfrac{1}{5}, \\
h(4) &= \dfrac{3}{10}, \\
h(5) &= \dfrac{3}{5}.
\end{aligned}
$$

The five leverages sum to $2$, which is $p$, not $n$. Leverage is not a probability attached to a single row. It is a diagonal entry of a projector, and only the trace has to equal the number of coefficients.


In [2]:
# Simple-regression leverages sum to the number of coefficients.
from fractions import Fraction
x = [1, 2, 3, 4, 5]
xbar, sxx, n = 3, 10, 5
leverages = [Fraction(1, n) + Fraction((xi - xbar) ** 2, sxx) for xi in x]
print("leverages", leverages)
print("sum", sum(leverages))
assert leverages == [Fraction(3, 5), Fraction(3, 10), Fraction(1, 5), Fraction(3, 10), Fraction(3, 5)]
assert sum(leverages) == 2


leverages [Fraction(3, 5), Fraction(3, 10), Fraction(1, 5), Fraction(3, 10), Fraction(3, 5)]
sum 2


## Pitfall

A high leverage does not by itself mean a bad observation. On the three-point line the first and last leverages are $5/6$, and those points are ordinary members of a short table. Leverage says how far $x$ sits from the center of the design. Influence, the subject of a later lesson, also needs the size of the residual.


In [3]:
# The largest leverage in this design belongs to the endpoints, residuals included.
from fractions import Fraction
leverages = [Fraction(5, 6), Fraction(1, 3), Fraction(5, 6)]
residuals = [Fraction(1, 6), Fraction(-1, 3), Fraction(1, 6)]
print("largest leverage", max(leverages))
print("residual at that same index", residuals[0])
assert max(leverages) == Fraction(5, 6)
assert residuals[0] != 0


largest leverage 5/6
residual at that same index 1/6


## Summary

- $\widehat y = Hy$ with $H = X(X^{\mathsf T}X)^{-1}X^{\mathsf T}$. Then $H^2 = H$ and $X^{\mathsf T}(y - \widehat y) = 0$.
- $\operatorname{tr}(H) = p$. With an intercept, each row of $H$ sums to $1$.
- In simple regression, $h_{ii} = 1/n + (x_i-\bar x)^2/S_{xx}$. Distance in $x$ raises leverage.
